# AgenticRAG-Bench — Week 4
## Query Rewriting + D4 Noise Robustness

**What's new in Week 4:**
1. Query rewriting — LLM rewrites each search query before FAISS to improve D2
2. D4 metric — noise robustness via cross-question paragraph injection
3. Two-phase evaluation: clean run → noisy run → interference rate

**What does NOT change from Week 3:**
- Agent: LangGraph ReAct + Llama 3.1 8B
- Embeddings: nomic-embed-text
- k = 3 | system prompt ON | max_steps = 10

**Week 3 baseline (what we're trying to beat):**
- D1 = 22% (11/50 correct)
- D2 = 0.183 (26/50 questions retrieved zero relevant docs)
- D3 = 0.626 (planning is not the bottleneck)
- D5 = 0.811

---

## Cell 0 — Environment + imports

In [1]:
import sys
import os
import json
import time
import math
import re
import copy
import random
import signal
from pathlib import Path
from difflib import SequenceMatcher
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
os.environ["RAGAS_DO_NOT_PARALLELIZE"] = "true"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Fix OMP libomp double-init crash with FAISS
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

import requests
r = requests.get("http://localhost:11434/api/tags", timeout=3)
models = [m['name'] for m in r.json().get('models', [])]
print("Ollama models:", models)

from datasets import load_dataset
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent

Path(PROJECT_ROOT / "data/questions").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "data/trajectories").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "notes").mkdir(parents=True, exist_ok=True)

print("All imports OK")

Project root: /Users/idhantsingh/Desktop/agenticrag-bench
Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]
Ollama models: ['nomic-embed-text:latest', 'llama3.1:8b']
All imports OK


---
## Cell 1 — All metric classes (D1, D2, D3, D4, D5)

D1, D2, D3, D5 unchanged from Week 3. **D4 is new.**

In [2]:
# ── D1: Answer Correctness ───────────────────────────────────────

class D1_AnswerCorrectness:
    name = "D1_answer_correctness"
    STOPWORDS = {
        "the","a","an","is","was","of","in","to","and","or",
        "it","its","be","been","being","have","has","had",
        "by","at","from","with","that","this","are"
    }
    LEGAL_SUFFIXES = {
        "inc","incorporated","corp","corporation",
        "ltd","limited","llc","plc","gmbh","company"
    }

    def _normalize(self, text):
        text = text.lower().strip()
        text = re.sub(r'[^\w\s]', ' ', text)
        words = [w for w in text.split() if w not in self.LEGAL_SUFFIXES]
        return ' '.join(words)

    def _meaningful_words(self, text):
        text = re.sub(r'[^\w\s]', ' ', text)
        return {w for w in text.split()
                if w not in self.STOPWORDS
                and w not in self.LEGAL_SUFFIXES
                and len(w) > 1}

    def _is_degenerate(self, predicted):
        """Detect degenerate outputs: raw tool dumps, refusals, empty."""
        p = predicted.strip()
        return (
            (p.startswith('{') and 'search_knowledge_base' in p)
            or p.lower().startswith('sorry')
            or len(p) < 5
        )

    def score(self, predicted, ground_truth):
        degenerate = self._is_degenerate(predicted)
        if degenerate:
            return {
                "exact_match": False, "near_match": False,
                "token_recall": 0.0, "seq_sim": 0.0,
                "score": 0.0, "degenerate_output": True
            }

        pred_raw  = predicted.lower().strip()
        truth_raw = ground_truth.lower().strip()
        exact_raw  = truth_raw in pred_raw
        pred_norm  = self._normalize(predicted)
        truth_norm = self._normalize(ground_truth)
        if ' ' not in truth_norm:
            exact_norm = (
                pred_norm == truth_norm or
                pred_norm.startswith(truth_norm + ' ') or
                (' is ' + truth_norm) in pred_norm or
                (' was ' + truth_norm) in pred_norm
            )
        else:
            exact_norm = truth_norm in pred_norm
        exact_match = exact_raw or exact_norm
        pred_words  = self._meaningful_words(pred_norm)
        truth_words = self._meaningful_words(truth_norm)
        if truth_words:
            recall = len(pred_words & truth_words) / len(truth_words)
        else:
            recall = 1.0 if exact_match else 0.0
        seq = SequenceMatcher(None, truth_norm,
                              pred_norm[:len(truth_norm)*3]).ratio()
        near = (recall >= 0.6) or (seq >= 0.7 and recall >= 0.4)
        if exact_match:    score = 1.0
        elif near:         score = round(0.5 + recall * 0.5, 3)
        else:
            intersection = pred_words & truth_words
            union        = pred_words | truth_words
            score = round(len(intersection)/len(union), 3) if union else 0.0
        return {
            "exact_match": exact_match, "near_match": near,
            "token_recall": round(recall,3), "seq_sim": round(seq,3),
            "score": score, "degenerate_output": False
        }




# ── D2: Retrieval Step Quality ───────────────────────────────────

class D2_RetrievalStepQuality:
    name = "D2_retrieval_step_quality"
    def _doc_is_relevant(self, doc_text, supporting_paragraphs):
        doc_lower = doc_text.lower()
        for sup in supporting_paragraphs:
            if sup[:80].lower() in doc_lower:
                return True
            sup_words = set(sup.lower().split())
            doc_words = set(doc_lower.split())
            if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.7:
                return True
        return False

    def score(self, trajectory_steps, supporting_paragraphs):
        if not trajectory_steps:
            return {"avg_precision_at_k":0.0, "best_step_precision":0.0,
                    "any_relevant_found":False, "step_precisions":[], "d2_score":0.0}
        step_precisions = []
        any_relevant = False
        for step in trajectory_steps:
            docs = step.get("retrieved_content", [])
            if not docs:
                step_precisions.append(0.0)
                continue
            hits = sum(1 for doc in docs if self._doc_is_relevant(doc, supporting_paragraphs))
            prec = hits / len(docs)
            step_precisions.append(round(prec, 3))
            if hits > 0: any_relevant = True
        avg_prec = sum(step_precisions) / len(step_precisions)
        best_prec = max(step_precisions)
        d2_score = round(min(1.0, avg_prec * 0.8 + (0.2 if any_relevant else 0.0)), 3)
        return {"avg_precision_at_k": round(avg_prec,3), "best_step_precision": round(best_prec,3),
                "any_relevant_found": any_relevant, "step_precisions": step_precisions, "d2_score": d2_score}




# ── D3: Planning Coherence ───────────────────────────────────────

class D3_PlanningCoherence:
    name = "D3_planning_coherence"
    def _query_diversity(self, queries):
        if len(queries) <= 1: return 0.0
        unique = list(set(queries))
        if len(unique) == 1: return 0.0
        pairs = []
        for i in range(len(unique)):
            for j in range(i+1, len(unique)):
                pairs.append(1.0 - SequenceMatcher(None, unique[i], unique[j]).ratio())
        return round(sum(pairs)/len(pairs), 3) if pairs else 0.0

    def score(self, trajectory_summary, num_hops):
        total_steps    = trajectory_summary.get("total_steps", 0)
        unique_queries = trajectory_summary.get("unique_queries", [])
        redundancy     = trajectory_summary.get("redundancy_rate", 0)
        loop_detected  = trajectory_summary.get("loop_detected", False)
        num_unique = len(unique_queries)
        hop_coverage = min(1.0, num_unique / max(num_hops, 1))
        diversity = self._query_diversity(unique_queries)
        loop_penalty      = 0.4 if loop_detected else 0.0
        undershoot        = max(0.0, (num_hops - num_unique) / max(num_hops, 1))
        undershoot_penalty = undershoot * 0.5
        raw = (hop_coverage * 0.5) + (diversity * 0.5) - loop_penalty - undershoot_penalty
        d3_score = round(max(0.0, min(1.0, raw)), 3)
        return {"num_hops_required": num_hops, "unique_queries_made": num_unique,
                "hop_coverage": round(hop_coverage,3), "query_diversity": round(diversity,3),
                "loop_penalty": loop_penalty, "undershoot_penalty": round(undershoot_penalty,3),
                "d3_score": d3_score}




# ── D4: Noise Robustness (NEW) ───────────────────────────────────

class D4_NoiseRobustness:
    name = "D4_noise_robustness"

    def score(self, clean_results, noisy_results):
        """
        Compare clean vs noisy runs for the same question set.
        Returns per-question and aggregate interference metrics.
        """
        correct_clean = sum(1 for r in clean_results if r['D1_answer_correctness']['exact_match'])
        correct_noisy = sum(1 for r in noisy_results if r['D1_answer_correctness']['exact_match'])

        if correct_clean == 0:
            interference_rate = None  # can't measure — 0% clean baseline
        else:
            interference_rate = round((correct_clean - correct_noisy) / correct_clean, 3)

        # Per-question breakdown
        per_question = []
        for c, n in zip(clean_results, noisy_results):
            c_correct = c['D1_answer_correctness']['exact_match']
            n_correct = n['D1_answer_correctness']['exact_match']
            per_question.append({
                "question_id": c["question_id"],
                "correct_clean": c_correct,
                "correct_noisy": n_correct,
                "flipped": c_correct and not n_correct,  # was right, now wrong
                "gained": not c_correct and n_correct,   # was wrong, now right (noise helped?)
                "d1_clean": c['D1_answer_correctness']['score'],
                "d1_noisy": n['D1_answer_correctness']['score'],
                "d2_clean": c['D2_retrieval_step_quality']['d2_score'],
                "d2_noisy": n['D2_retrieval_step_quality']['d2_score'],
            })

        flipped_count = sum(1 for p in per_question if p["flipped"])
        gained_count  = sum(1 for p in per_question if p["gained"])

        return {
            "correct_clean": correct_clean,
            "correct_noisy": correct_noisy,
            "interference_rate": interference_rate,
            "flipped_count": flipped_count,
            "gained_count": gained_count,
            "per_question": per_question,
        }




# ── D5: Trajectory Efficiency ────────────────────────────────────

class D5_TrajectoryEfficiency:
    name = "D5_trajectory_efficiency"
    BASELINE_TOKENS_PER_STEP = 400
    def score(self, trajectory_summary, answer_correct, degenerate=False):
        if degenerate:
            return {"total_steps": 0, "redundancy_rate": 0,
                    "tokens_per_step": 0, "loop_detected": False,
                    "loop_penalty_applied": 0, "efficiency_score": 0.0,
                    "total_tokens": 0, "tokens_per_correct_answer": None,
                    "degenerate": True}

        total_steps    = trajectory_summary.get("total_steps", 0)
        redundancy     = trajectory_summary.get("redundancy_rate", 0)
        tokens_per_step= trajectory_summary.get("tokens_per_step", 0)
        loop_detected  = trajectory_summary.get("loop_detected", False)
        total_tokens   = trajectory_summary.get("total_tokens", 0)
        token_penalty  = min(1.0, tokens_per_step / (self.BASELINE_TOKENS_PER_STEP * 2))
        loop_penalty   = 0.3 if loop_detected else 0.0
        raw  = 1.0 - (redundancy*0.5) - (token_penalty*0.3) - loop_penalty
        eff  = max(0.0, round(raw, 3))
        return {"total_steps": total_steps, "redundancy_rate": redundancy,
                "tokens_per_step": tokens_per_step, "loop_detected": loop_detected,
                "loop_penalty_applied": loop_penalty, "efficiency_score": eff,
                "total_tokens": total_tokens,
                "tokens_per_correct_answer": total_tokens if answer_correct else None}


# ── Quick tests ──────────────────────────────────────────────────

d1 = D1_AnswerCorrectness()
d2 = D2_RetrievalStepQuality()
d3 = D3_PlanningCoherence()
d4 = D4_NoiseRobustness()
d5 = D5_TrajectoryEfficiency()

# D1 tests
assert d1.score("Bombardier Aerospace made it", "Bombardier Inc.")["near_match"] == True
assert d1.score("Miquette Giraudy is her name", "Miquette Giraudy")["exact_match"] == True
assert d1.score("I don't know", "Paris")["score"] == 0.0
print("D1 tests passed")

# D2 test
fake_steps = [
    {"retrieved_content": ["Steve Hillage is a British musician and spouse of Miquette Giraudy",
                            "The Godfather was directed by Coppola",
                            "Churchill was Prime Minister"]},
]
d2_result = d2.score(fake_steps, ["Steve Hillage is a British musician and spouse of Miquette Giraudy"])
assert d2_result["any_relevant_found"] == True
assert d2_result["avg_precision_at_k"] == round(1/3, 3)
print("D2 tests passed")

# D3 test
traj_good = {"total_steps":2, "unique_queries":["Steve Hillage musician","Steve Hillage spouse"],
             "redundancy_rate":0.0, "loop_detected":False}
traj_bad  = {"total_steps":2, "unique_queries":["Green performer spouse"],
             "redundancy_rate":0.5, "loop_detected":True}
d3_good = d3.score(traj_good, num_hops=2)
d3_bad  = d3.score(traj_bad,  num_hops=2)
assert d3_good["d3_score"] > d3_bad["d3_score"], "Good planning should score higher"
print(f"D3 tests passed — good={d3_good['d3_score']}, bad={d3_bad['d3_score']}")

# D4 test
fake_clean = [
    {"question_id": "q1", "D1_answer_correctness": {"exact_match": True, "score": 1.0},
     "D2_retrieval_step_quality": {"d2_score": 0.467}},
    {"question_id": "q2", "D1_answer_correctness": {"exact_match": True, "score": 1.0},
     "D2_retrieval_step_quality": {"d2_score": 0.333}},
    {"question_id": "q3", "D1_answer_correctness": {"exact_match": False, "score": 0.0},
     "D2_retrieval_step_quality": {"d2_score": 0.0}},
]
fake_noisy = [
    {"question_id": "q1", "D1_answer_correctness": {"exact_match": False, "score": 0.0},
     "D2_retrieval_step_quality": {"d2_score": 0.0}},   # flipped
    {"question_id": "q2", "D1_answer_correctness": {"exact_match": True, "score": 1.0},
     "D2_retrieval_step_quality": {"d2_score": 0.333}},  # survived
    {"question_id": "q3", "D1_answer_correctness": {"exact_match": False, "score": 0.0},
     "D2_retrieval_step_quality": {"d2_score": 0.0}},    # still wrong
]
d4_result = d4.score(fake_clean, fake_noisy)
assert d4_result["correct_clean"] == 2
assert d4_result["correct_noisy"] == 1
assert d4_result["interference_rate"] == 0.5
assert d4_result["flipped_count"] == 1
print(f"D4 tests passed — interference_rate={d4_result['interference_rate']}")

print("\nAll metric classes ready")

D1 tests passed
D2 tests passed
D3 tests passed — good=0.619, bad=0.0
D4 tests passed — interference_rate=0.5

All metric classes ready


---
## Cell 2 — TrajectoryLogger (updated: stores rewritten_query)

In [3]:

class TrajectoryLogger:
    def __init__(self, max_steps=10):
        self.max_steps = max_steps
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def reset(self):
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def log(self, query, retrieved_docs, latency_ms, supporting_paragraphs=None, rewritten_query=None):
        is_loop = bool(self.steps and self.steps[-1]['query'] == query)
        if is_loop:
            self.loop_detected = True
            self.loop_query = query
        precision_at_k = 0.0
        if supporting_paragraphs and retrieved_docs:
            def _is_relevant(doc):
                doc_words = set(doc.lower().split())
                for sup in supporting_paragraphs:
                    sup_words = set(sup.lower().split())
                    if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.5:
                        return True
                return False
            hits = sum(1 for doc in retrieved_docs if _is_relevant(doc))
            precision_at_k = round(hits / len(retrieved_docs), 3)
        tokens_estimate = (len(query) + sum(len(d) for d in retrieved_docs)) // 4
        step = {
            "step_id":            len(self.steps) + 1,
            "query":              query,
            "rewritten_query":    rewritten_query,
            "num_docs_retrieved": len(retrieved_docs),
            "retrieved_content":  retrieved_docs,
            "precision_at_k":     precision_at_k,
            "is_loop_step":       is_loop,
            "tokens_estimate":    tokens_estimate,
            "latency_ms":         round(latency_ms, 1)
        }
        self.steps.append(step)
        return step

    def at_limit(self):
        return len(self.steps) >= self.max_steps

    def summary(self):
        if not self.steps: return {}
        n = len(self.steps)
        unique = list(set(s['query'] for s in self.steps))
        redundant = n - len(unique)
        total_tokens = sum(s['tokens_estimate'] for s in self.steps)
        avg_prec = sum(s['precision_at_k'] for s in self.steps) / n
        return {
            "total_steps": n, "unique_queries": unique,
            "redundant_queries": redundant,
            "redundancy_rate": round(redundant / n, 3),
            "loop_detected": self.loop_detected,
            "loop_steps": sum(1 for s in self.steps if s['is_loop_step']),
            "avg_step_precision_at_k": round(avg_prec, 3),
            "total_tokens": total_tokens,
            "total_latency_ms": round(sum(s['latency_ms'] for s in self.steps), 1),
            "tokens_per_step": round(total_tokens / n, 1)
        }
print('TrajectoryLogger ready')


TrajectoryLogger ready


---
## Cell 3 — Load 50 questions (from Week 3 dataset)

In [4]:
q_path = PROJECT_ROOT / "data/questions/3_musique_50.json"
with open(q_path) as f:
    questions = json.load(f)

print(f"Loaded {len(questions)} questions from {q_path}")
print(f"  Avg KB size per question:    {sum(len(q['paragraphs']) for q in questions)//len(questions)} docs")
print(f"  Avg supporting docs:         {sum(q['num_supporting'] for q in questions)/len(questions):.1f}")
print(f"  Avg distractors:             {sum(q['num_distractors'] for q in questions)/len(questions):.1f}")

Loaded 50 questions from /Users/idhantsingh/Desktop/agenticrag-bench/data/questions/3_musique_50.json
  Avg KB size per question:    20 docs
  Avg supporting docs:         2.0
  Avg distractors:             18.0


---
## Cell 4 — Noise injection function (cross-question paragraph swap)

In [5]:
def inject_noise(questions, noise_fraction=0.25, seed=42):
    """
    Cross-question swap: replace `noise_fraction` of each question's
    distractor paragraphs with supporting paragraphs from other questions.
    
    This creates plausible interference — the injected paragraphs are real
    Wikipedia text about similar entity types, but contain facts that
    answer DIFFERENT questions. A fragile agent may latch onto these.
    
    Returns a new list of questions with modified paragraph sets.
    Supporting paragraphs are NEVER replaced.
    """
    rng = random.Random(seed)
    
    # Collect all supporting paragraphs from all questions, indexed by question
    all_supporting = []
    for q in questions:
        for sp in q['supporting_paragraphs']:
            all_supporting.append(sp)
    
    noisy_questions = []
    total_replaced = 0
    
    for q in questions:
        q_copy = copy.deepcopy(q)
        distractors = q_copy['distractor_paragraphs']
        n_replace = max(1, int(len(distractors) * noise_fraction))
        
        # Pick random distractor positions to replace
        replace_indices = rng.sample(range(len(distractors)), min(n_replace, len(distractors)))
        
        # Pick replacement paragraphs from OTHER questions' supporting docs
        other_supporting = [s for s in all_supporting if s not in q['supporting_paragraphs']]
        replacements = rng.sample(other_supporting, min(n_replace, len(other_supporting)))
        
        for idx, replacement in zip(replace_indices, replacements):
            distractors[idx] = replacement
        
        # Rebuild full paragraph list (supporting + modified distractors)
        q_copy['paragraphs'] = q_copy['supporting_paragraphs'] + distractors
        rng.shuffle(q_copy['paragraphs'])
        q_copy['noise_injected'] = True
        q_copy['n_paragraphs_replaced'] = len(replacements)
        total_replaced += len(replacements)
        noisy_questions.append(q_copy)
    
    return noisy_questions


# ── Test noise injection ─────────────────────────────────────────

noisy_q = inject_noise(questions, noise_fraction=0.25, seed=42)

# Verify supporting paragraphs are preserved
for orig, noisy in zip(questions, noisy_q):
    for sp in orig['supporting_paragraphs']:
        assert sp in noisy['paragraphs'], f"Supporting paragraph lost in {orig['id']}"

# Stats
avg_replaced = sum(q['n_paragraphs_replaced'] for q in noisy_q) / len(noisy_q)
print(f"Noise injection test passed:")
print(f"  Questions: {len(noisy_q)}")
print(f"  Avg paragraphs replaced per question: {avg_replaced:.1f}")
print(f"  Avg KB size (noisy): {sum(len(q['paragraphs']) for q in noisy_q)//len(noisy_q)} docs")
print(f"  Supporting paragraphs preserved: ✓")

Noise injection test passed:
  Questions: 50
  Avg paragraphs replaced per question: 4.0
  Avg KB size (noisy): 20 docs
  Supporting paragraphs preserved: ✓


---
## Cell 5 — Build agent (query rewriting ON)

In [6]:
print("Loading embedding model...")
embedding_model = OllamaEmbeddings(model="nomic-embed-text")
_ = embedding_model.embed_query("test")
print("nomic-embed-text OK")

TRAJECTORY_LOGGER = TrajectoryLogger(max_steps=10)
current_context = {"store": None, "supporting_paragraphs": []}

MAX_CHARS_PER_DOC = 500  # Truncate tool output for LLM context

llm = ChatOllama(model="llama3.1:8b", temperature=0)


@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the knowledge base for information relevant to the query.
    For multi-hop questions ALWAYS search at least twice:
    - First: find what the subject IS
    - Second: find the specific fact about that subject
    Never repeat the same query. If results are unhelpful, change the query.
    """
    if TRAJECTORY_LOGGER.at_limit():
        return "[SEARCH LIMIT REACHED: provide your best answer now.]"

    # Query Rewriting: ask the LLM to generate a cleaner search query
    rewrite_prompt = (
        "Rewrite this search query to be effective for a vector similarity search. "
        "Extract key entities and facts. Remove filler words. "
        "Output ONLY the rewritten query, nothing else.\n"
        f"Original query: {query}"
    )
    try:
        rewritten = llm.invoke([("user", rewrite_prompt)]).content.strip().strip('\"\'')
    except Exception:
        rewritten = query

    t0 = time.time()
    results = current_context["store"].similarity_search(rewritten, k=3)
    latency = (time.time() - t0) * 1000
    texts = [r.page_content for r in results]

    step = TRAJECTORY_LOGGER.log(
        query, texts, latency,
        current_context["supporting_paragraphs"],
        rewritten_query=rewritten
    )

    # Truncate for LLM context — full text is already saved in trajectory
    truncated = [t[:MAX_CHARS_PER_DOC] for t in texts]

    warning = ""
    if step["is_loop_step"]:
        warning = "\n\n[You already searched this query. Use a DIFFERENT query.]"

    # Nudge agent to keep searching after first step
    nudge = ""
    if len(TRAJECTORY_LOGGER.steps) == 1:
        nudge = "\n\n[NOTE: You have completed 1 search. Multi-hop questions require at least 2 searches. Search again with a DIFFERENT query before answering.]"

    return "\n\n---\n\n".join(truncated) + warning + nudge


SYSTEM_PROMPT = """You are a research assistant answering multi-hop questions.
These questions require combining facts from 2+ sources.

RULES:
1. ALWAYS search at least twice before answering.
2. First search: find what the main entity IS.
3. Second search: find the specific fact about that entity.
4. Never repeat a query — vary your searches.
5. Answer concisely after at least 2 searches."""

agent = create_react_agent(llm, [search_knowledge_base], prompt=SYSTEM_PROMPT)

print("Agent ready — k=3, system prompt ON, query rewriting ON, max_steps=10")

Loading embedding model...
nomic-embed-text OK
Agent ready — k=3, system prompt ON, query rewriting ON, max_steps=10


/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_27269/3282108882.py:74: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [search_knowledge_base], prompt=SYSTEM_PROMPT)


---
## Cell 6 — Run function with timeout + recursion_limit

In [7]:
d1_metric = D1_AnswerCorrectness()
d2_metric = D2_RetrievalStepQuality()
d3_metric = D3_PlanningCoherence()
d5_metric = D5_TrajectoryEfficiency()

# Timeout wrapper
class QuestionTimeoutError(Exception):
    pass

def _timeout_handler(signum, frame):
    raise QuestionTimeoutError("Question timed out")

QUESTION_TIMEOUT = 120  # seconds per question


def run_question(q):
    global current_context
    TRAJECTORY_LOGGER.reset()

    # Build per-question FAISS store
    docs  = [Document(page_content=p) for p in q['paragraphs']]
    store = FAISS.from_documents(docs, embedding_model)
    current_context["store"]                = store
    current_context["supporting_paragraphs"]= q['supporting_paragraphs']

    t0 = time.time()

    # Timeout so one question can't hang forever
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(QUESTION_TIMEOUT)

    try:
        result  = agent.invoke(
            {"messages": [("user", q["question"])]},
            config={"recursion_limit": 28}
        )
        answer  = result["messages"][-1].content
        success = True
    except QuestionTimeoutError:
        answer  = "TIMEOUT"
        success = False
        print(f"    ⚠ TIMEOUT after {QUESTION_TIMEOUT}s")
    except Exception as e:
        answer  = f"ERROR: {e}"
        success = False
        print(f"    ⚠ ERROR: {e}")
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    wall_ms   = (time.time() - t0) * 1000
    traj_sum  = TRAJECTORY_LOGGER.summary()

    d1 = d1_metric.score(answer, q["answer"])
    d2 = d2_metric.score(TRAJECTORY_LOGGER.steps, q["supporting_paragraphs"])
    d3 = d3_metric.score(traj_sum, q["num_hops"])
    d5 = d5_metric.score(traj_sum, d1["exact_match"], degenerate=d1.get("degenerate_output", False))

    return {
        "question_id":        q["id"],
        "question":           q["question"],
        "ground_truth":       q["answer"],
        "num_hops":           q["num_hops"],
        "difficulty":         q["difficulty"],
        "predicted_answer":   answer,
        "success":            success,
        "wall_latency_ms":    round(wall_ms, 1),
        "noise_injected":     q.get("noise_injected", False),
        "trajectory":         TRAJECTORY_LOGGER.steps.copy(),
        "trajectory_summary": traj_sum,
        "D1_answer_correctness":     d1,
        "D2_retrieval_step_quality": d2,
        "D3_planning_coherence":     d3,
        "D5_trajectory_efficiency":  d5,
    }

print("Run function ready — scores D1, D2, D3, D5 per question")

Run function ready — scores D1, D2, D3, D5 per question


---
## Cell 7 — Phase A: Clean run (50 questions, query rewriting ON)

Each question takes ~10–50s. Total ~10–15 minutes. Results saved after every question.

In [8]:
N_RUN = 50
to_run = questions[:N_RUN]
clean_results = []

print(f"PHASE A — Clean run: {N_RUN} questions, query rewriting ON")
print("k=3 | system prompt ON | nomic-embed-text | max_steps=10")
print(f"recursion_limit=28 | timeout={QUESTION_TIMEOUT}s per question")
print("=" * 65)

for i, q in enumerate(to_run):
    print(f"\nQ{i+1}/{N_RUN}: {q['question'][:65]}...")
    print(f"  Expected: {q['answer']} | Hops: {q['num_hops']} | "
          f"RC={q['difficulty']['reasoning_complexity']} "
          f"RD={q['difficulty']['retrieval_difficulty']}")

    r = run_question(q)
    clean_results.append(r)

    print(f"  Predicted: {r['predicted_answer'][:70]}")
    print(f"  D1={r['D1_answer_correctness']['score']:.3f}  "
          f"D2={r['D2_retrieval_step_quality']['d2_score']:.3f}  "
          f"D3={r['D3_planning_coherence']['d3_score']:.3f}  "
          f"D5={r['D5_trajectory_efficiency']['efficiency_score']:.3f}  "
          f"({r['wall_latency_ms']/1000:.0f}s)")

    # Rewritten query info
    for step in r['trajectory']:
        if step.get('rewritten_query') and step['rewritten_query'] != step['query']:
            print(f"    Rewrite: '{step['query'][:40]}' → '{step['rewritten_query'][:40]}'")

    # Save incrementally
    out_path = PROJECT_ROOT / "data/trajectories/4_agent_results_clean.json"
    with open(out_path, "w") as f:
        json.dump(clean_results, f, indent=2)

# Quick summary
correct = sum(1 for r in clean_results if r['D1_answer_correctness']['exact_match'])
avg_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in clean_results) / len(clean_results)
print(f"\n{'=' * 65}")
print(f"PHASE A COMPLETE — Clean run")
print(f"  Accuracy: {correct}/{N_RUN} = {correct/N_RUN:.0%}")
print(f"  Avg D2:   {avg_d2:.3f}  (Week 3 baseline: 0.183)")
print(f"  Saved to {out_path}")

PHASE A — Clean run: 50 questions, query rewriting ON
k=3 | system prompt ON | nomic-embed-text | max_steps=10
recursion_limit=28 | timeout=120s per question

Q1/50: Who is the spouse of the Green performer?...
  Expected: Miquette Giraudy | Hops: 2 | RC=2 RD=3
  Predicted: Based on your query "Who is the spouse of the Green performer?", I wil
  D1=0.000  D2=0.467  D3=0.821  D5=0.917  (23s)
    Rewrite: 'What is the name of the lead singer of t' → 'Find songs by Green Day with Billie Joe '
    Rewrite: 'Who is Billie Joe Armstrong's spouse?' → 'Billie Joe Armstrong spouse'

Q2/50: Who founded the company that distributed the film UHF?...
  Expected: Mike Medavoy | Hops: 2 | RC=2 RD=3
  Predicted: The founder of the company that distributed the film "UHF" is Kevin Sm
  D1=0.000  D2=0.000  D3=0.783  D5=0.898  (12s)
    Rewrite: 'founder of company distributing UHF film' → 'Founder of company that distributes UHF '
    Rewrite: 'UHF film distributor' → 'UHF film distribution companies'

Q

---
## Cell 8 — Phase B: Noisy run (50 questions, 25% noise injected)

Same agent, same settings, but each question's KB has ~25% of distractors swapped with supporting paragraphs from other questions.

In [9]:
noisy_questions = inject_noise(questions[:N_RUN], noise_fraction=0.25, seed=42)
noisy_results = []

print(f"PHASE B — Noisy run: {N_RUN} questions, 25% noise injected")
print("k=3 | system prompt ON | nomic-embed-text | query rewriting ON")
print(f"recursion_limit=28 | timeout={QUESTION_TIMEOUT}s per question")
print("=" * 65)

for i, q in enumerate(noisy_questions):
    print(f"\nQ{i+1}/{N_RUN}: {q['question'][:65]}...")
    print(f"  Expected: {q['answer']} | Noise: {q['n_paragraphs_replaced']} paras replaced")

    r = run_question(q)
    noisy_results.append(r)

    print(f"  Predicted: {r['predicted_answer'][:70]}")
    print(f"  D1={r['D1_answer_correctness']['score']:.3f}  "
          f"D2={r['D2_retrieval_step_quality']['d2_score']:.3f}  "
          f"D3={r['D3_planning_coherence']['d3_score']:.3f}  "
          f"D5={r['D5_trajectory_efficiency']['efficiency_score']:.3f}  "
          f"({r['wall_latency_ms']/1000:.0f}s)")

    # Save incrementally
    out_path = PROJECT_ROOT / "data/trajectories/4_agent_results_noisy.json"
    with open(out_path, "w") as f:
        json.dump(noisy_results, f, indent=2)

# Quick summary
correct_noisy = sum(1 for r in noisy_results if r['D1_answer_correctness']['exact_match'])
avg_d2_noisy = sum(r['D2_retrieval_step_quality']['d2_score'] for r in noisy_results) / len(noisy_results)
print(f"\n{'=' * 65}")
print(f"PHASE B COMPLETE — Noisy run")
print(f"  Accuracy: {correct_noisy}/{N_RUN} = {correct_noisy/N_RUN:.0%}")
print(f"  Avg D2:   {avg_d2_noisy:.3f}")
print(f"  Saved to {out_path}")

PHASE B — Noisy run: 50 questions, 25% noise injected
k=3 | system prompt ON | nomic-embed-text | query rewriting ON
recursion_limit=28 | timeout=120s per question

Q1/50: Who is the spouse of the Green performer?...
  Expected: Miquette Giraudy | Noise: 4 paras replaced
  Predicted: Based on the information found, it appears that there are multiple ind
  D1=0.000  D2=0.467  D3=0.821  D5=0.917  (21s)

Q2/50: Who founded the company that distributed the film UHF?...
  Expected: Mike Medavoy | Noise: 4 paras replaced
  Predicted: The founder of the company that distributed the film UHF is Steven Spi
  D1=0.000  D2=0.333  D3=0.715  D5=0.885  (22s)

Q3/50: What administrative territorial entity is the owner of Ciudad Dep...
  Expected: Tamaulipas | Noise: 4 paras replaced
  Predicted: Based on the searches, I can conclude that the administrative territor
  D1=0.000  D2=0.333  D3=0.707  D5=0.912  (20s)

Q4/50: Where is Ulrich Walter's employer headquartered?...
  Expected: Cologne | Noise: 

---
## Cell 9 — D4 interference analysis

In [10]:
# Load both result sets
with open(PROJECT_ROOT / "data/trajectories/4_agent_results_clean.json") as f:
    clean_results = json.load(f)
with open(PROJECT_ROOT / "data/trajectories/4_agent_results_noisy.json") as f:
    noisy_results = json.load(f)

d4_metric = D4_NoiseRobustness()
d4_result = d4_metric.score(clean_results, noisy_results)

print("=" * 80)
print("D4 — NOISE ROBUSTNESS ANALYSIS")
print("=" * 80)
print()

# Summary
print(f"  Correct (clean):   {d4_result['correct_clean']}/{len(clean_results)}")
print(f"  Correct (noisy):   {d4_result['correct_noisy']}/{len(noisy_results)}")
print(f"  Interference rate: {d4_result['interference_rate']}")
print(f"  Flipped (right→wrong): {d4_result['flipped_count']}")
print(f"  Gained  (wrong→right): {d4_result['gained_count']}")
print()

# Per-question detail table
print(f"{'Q':3} {'Clean_D1':10} {'Noisy_D1':10} {'Flipped':8} {'Clean_D2':10} {'Noisy_D2':10}")
print("-" * 55)

for i, pq in enumerate(d4_result['per_question']):
    flip_marker = "YES ←" if pq['flipped'] else ("GAIN" if pq['gained'] else "")
    print(
        f"{i+1:3} "
        f"{pq['d1_clean']:10.3f} "
        f"{pq['d1_noisy']:10.3f} "
        f"{flip_marker:8} "
        f"{pq['d2_clean']:10.3f} "
        f"{pq['d2_noisy']:10.3f}"
    )

print("-" * 55)

# Interpretation
print()
if d4_result['interference_rate'] is not None:
    if d4_result['interference_rate'] >= 0.5:
        print("⚠ HIGH INTERFERENCE — noise destroys ≥50% of correct answers.")
        print("  The agent is highly fragile to corpus perturbation.")
    elif d4_result['interference_rate'] >= 0.2:
        print("⚡ MODERATE INTERFERENCE — noise flips some correct answers.")
    else:
        print("✓ LOW INTERFERENCE — agent is relatively robust to noise.")
else:
    print("⚠ Cannot compute interference rate — 0 correct answers on clean data.")

# Save D4 results
d4_out_path = PROJECT_ROOT / "data/trajectories/4_d4_interference.json"
with open(d4_out_path, "w") as f:
    json.dump(d4_result, f, indent=2)
print(f"\nSaved D4 results to {d4_out_path}")

D4 — NOISE ROBUSTNESS ANALYSIS

  Correct (clean):   15/50
  Correct (noisy):   17/50
  Interference rate: -0.133
  Flipped (right→wrong): 4
  Gained  (wrong→right): 6

Q   Clean_D1   Noisy_D1   Flipped  Clean_D2   Noisy_D2  
-------------------------------------------------------
  1      0.000      0.000               0.467      0.467
  2      0.000      0.000               0.000      0.333
  3      0.000      0.000               0.333      0.333
  4      0.000      0.000               0.000      0.000
  5      1.000      1.000               0.467      0.467
  6      1.000      0.026 YES ←         0.600      0.466
  7      0.014      0.000               0.466      0.466
  8      1.000      1.000               0.466      0.466
  9      0.015      0.000               0.333      0.000
 10      0.056      0.053               0.333      0.466
 11      0.000      0.000               0.000      0.000
 12      0.000      0.000               0.000      0.000
 13      1.000      1.000         

---
## Cell 10 — Full results comparison table

In [11]:
with open(PROJECT_ROOT / "data/trajectories/4_agent_results_clean.json") as f:
    clean_results = json.load(f)
with open(PROJECT_ROOT / "data/trajectories/4_agent_results_noisy.json") as f:
    noisy_results = json.load(f)

n = len(clean_results)

print("=" * 90)
print("WEEK 4 — FULL RESULTS (CLEAN vs NOISY)")
print("=" * 90)
print(f"{'Q':3} {'Hops':5} {'Clean_D1':9} {'Clean_D2':9} {'Clean_D3':9} {'Clean_D5':9} | {'Noisy_D1':9} {'Noisy_D2':9} {'Flip':5}")
print("-" * 90)

for i in range(n):
    c = clean_results[i]
    nr = noisy_results[i]
    c_correct = c['D1_answer_correctness']['exact_match']
    n_correct = nr['D1_answer_correctness']['exact_match']
    flip = "YES" if (c_correct and not n_correct) else ""
    print(
        f"{i+1:3} "
        f"{c['num_hops']:5} "
        f"{c['D1_answer_correctness']['score']:9.3f} "
        f"{c['D2_retrieval_step_quality']['d2_score']:9.3f} "
        f"{c['D3_planning_coherence']['d3_score']:9.3f} "
        f"{c['D5_trajectory_efficiency']['efficiency_score']:9.3f} | "
        f"{nr['D1_answer_correctness']['score']:9.3f} "
        f"{nr['D2_retrieval_step_quality']['d2_score']:9.3f} "
        f"{flip:5}"
    )

print("-" * 90)

# Aggregates — clean
correct_clean  = sum(1 for r in clean_results if r['D1_answer_correctness']['exact_match'])
degen_clean    = sum(1 for r in clean_results if r['D1_answer_correctness'].get('degenerate_output', False))
non_degen_clean = [r for r in clean_results if not r['D1_answer_correctness'].get('degenerate_output', False)]
n_nd = len(non_degen_clean)

avg_d1_c = sum(r['D1_answer_correctness']['score'] for r in clean_results) / n
avg_d2_c = sum(r['D2_retrieval_step_quality']['d2_score'] for r in non_degen_clean) / n_nd if n_nd else 0
avg_d3_c = sum(r['D3_planning_coherence']['d3_score'] for r in non_degen_clean) / n_nd if n_nd else 0
avg_d5_c = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in non_degen_clean) / n_nd if n_nd else 0
avg_steps_c = sum(r['trajectory_summary'].get('total_steps', 0) for r in non_degen_clean) / n_nd if n_nd else 0
loops_clean = sum(1 for r in clean_results if r['trajectory_summary'].get('loop_detected'))

# Aggregates — noisy
correct_noisy = sum(1 for r in noisy_results if r['D1_answer_correctness']['exact_match'])
avg_d1_n = sum(r['D1_answer_correctness']['score'] for r in noisy_results) / n
avg_d2_n = sum(r['D2_retrieval_step_quality']['d2_score'] for r in noisy_results) / n

# Tokens per correct answer (clean)
correct_tokens = [
    r['D5_trajectory_efficiency']['tokens_per_correct_answer']
    for r in clean_results
    if r['D1_answer_correctness']['exact_match']
    and r['D5_trajectory_efficiency']['tokens_per_correct_answer'] is not None
]
avg_tpca = sum(correct_tokens) / len(correct_tokens) if correct_tokens else 0

# D4
d4_metric = D4_NoiseRobustness()
d4_result = d4_metric.score(clean_results, noisy_results)

print(f"\nAGGREGATES ({n} questions):")
print(f"")
print(f"  CLEAN RUN ({degen_clean} degenerate excluded from D2/D3/D5):")
print(f"    Accuracy (D1 exact):   {correct_clean}/{n} = {correct_clean/n:.0%}  (Week 3: 22%)")
print(f"    Avg D1 score:          {avg_d1_c:.3f}")
print(f"    Avg D2 score:          {avg_d2_c:.3f}  (Week 3: 0.183)")
print(f"    Avg D3 score:          {avg_d3_c:.3f}  (Week 3: 0.626)")
print(f"    Avg D5 efficiency:     {avg_d5_c:.3f}")
print(f"    Avg steps/question:    {avg_steps_c:.1f}")
print(f"    Loops detected:        {loops_clean}/{n}")
print(f"    Tokens/correct answer: {avg_tpca:.1f}")
print(f"")
print(f"  NOISY RUN:")
print(f"    Accuracy (D1 exact):   {correct_noisy}/{n} = {correct_noisy/n:.0%}")
print(f"    Avg D1 score:          {avg_d1_n:.3f}")
print(f"    Avg D2 score:          {avg_d2_n:.3f}")
print(f"")
print(f"  D4 NOISE ROBUSTNESS:")
print(f"    Interference rate:     {d4_result['interference_rate']}")
print(f"    Flipped (right→wrong): {d4_result['flipped_count']}")
print(f"    Gained  (wrong→right): {d4_result['gained_count']}")

WEEK 4 — FULL RESULTS (CLEAN vs NOISY)
Q   Hops  Clean_D1  Clean_D2  Clean_D3  Clean_D5  | Noisy_D1  Noisy_D2  Flip 
------------------------------------------------------------------------------------------
  1     2     0.000     0.467     0.821     0.917 |     0.000     0.467      
  2     2     0.000     0.000     0.783     0.898 |     0.000     0.333      
  3     2     0.000     0.333     0.707     0.912 |     0.000     0.333      
  4     2     0.000     0.000     0.909     0.887 |     0.000     0.000      
  5     2     1.000     0.467     0.797     0.888 |     1.000     0.467      
  6     2     1.000     0.600     0.785     0.884 |     0.026     0.466 YES  
  7     2     0.014     0.466     0.536     0.849 |     0.000     0.466      
  8     2     1.000     0.466     0.203     0.461 |     1.000     0.466      
  9     2     0.015     0.333     0.545     0.850 |     0.000     0.000      
 10     2     0.056     0.333     0.776     0.916 |     0.053     0.466      
 11     2   

---
## Cell 11 — Week 4 summary + observations

In [12]:
with open(PROJECT_ROOT / "data/trajectories/4_agent_results_clean.json") as f:
    clean_results = json.load(f)
with open(PROJECT_ROOT / "data/trajectories/4_agent_results_noisy.json") as f:
    noisy_results = json.load(f)
with open(PROJECT_ROOT / "data/trajectories/4_d4_interference.json") as f:
    d4_result = json.load(f)

n = len(clean_results)
correct_clean  = sum(1 for r in clean_results if r['D1_answer_correctness']['exact_match'])
correct_noisy  = sum(1 for r in noisy_results if r['D1_answer_correctness']['exact_match'])
degen_clean    = sum(1 for r in clean_results if r['D1_answer_correctness'].get('degenerate_output', False))
non_degen      = [r for r in clean_results if not r['D1_answer_correctness'].get('degenerate_output', False)]
n_nd = len(non_degen)

avg_d1_c = sum(r['D1_answer_correctness']['score'] for r in clean_results) / n
avg_d2_c = sum(r['D2_retrieval_step_quality']['d2_score'] for r in non_degen) / n_nd if n_nd else 0
avg_d3_c = sum(r['D3_planning_coherence']['d3_score'] for r in non_degen) / n_nd if n_nd else 0
avg_d5_c = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in non_degen) / n_nd if n_nd else 0
avg_steps_c = sum(r['trajectory_summary'].get('total_steps', 0) for r in non_degen) / n_nd if n_nd else 0
loops_clean = sum(1 for r in clean_results if r['trajectory_summary'].get('loop_detected'))

avg_d1_n = sum(r['D1_answer_correctness']['score'] for r in noisy_results) / n
avg_d2_n = sum(r['D2_retrieval_step_quality']['d2_score'] for r in noisy_results) / n

correct_tokens = [
    r['D5_trajectory_efficiency']['tokens_per_correct_answer']
    for r in clean_results
    if r['D1_answer_correctness']['exact_match']
    and r['D5_trajectory_efficiency']['tokens_per_correct_answer'] is not None
]
avg_tpca = sum(correct_tokens) / len(correct_tokens) if correct_tokens else 0

# Correct/incorrect D2 split
correct_runs = [r for r in clean_results if r['D1_answer_correctness']['exact_match']]
incorrect_runs = [r for r in non_degen if not r['D1_answer_correctness']['exact_match']]
corr_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in correct_runs)/len(correct_runs) if correct_runs else 0
incorr_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in incorrect_runs)/len(incorrect_runs) if incorrect_runs else 0

interference_rate = d4_result.get('interference_rate', 'N/A')
flipped = d4_result.get('flipped_count', 0)
gained = d4_result.get('gained_count', 0)

notes = f"""# Week 4 Observations — AgenticRAG-Bench
Date: {time.strftime('%Y-%m-%d')}

## What was built
- Query rewriting: LLM rewrites search queries before FAISS for better vector similarity
- D4 noise robustness metric implemented and validated
- Cross-question paragraph swap noise injection (25% of distractors replaced)
- Two-phase evaluation: clean run + noisy run on same 50 questions
- TrajectoryLogger updated to store rewritten_query per step

## Clean run results ({n} questions, query rewriting ON)
- Accuracy:       {correct_clean}/{n} = {correct_clean/n:.0%}  (Week 3: 22%)
- Avg D1 score:   {avg_d1_c:.3f}
- Avg D2 score:   {avg_d2_c:.3f}  (Week 3: 0.183)
- Avg D3 score:   {avg_d3_c:.3f}  (Week 3: 0.626)
- Avg D5 score:   {avg_d5_c:.3f}
- Avg steps/q:    {avg_steps_c:.2f}
- Loops detected: {loops_clean}/{n}
- Degenerate:     {degen_clean}/{n}
- Tokens/correct: {avg_tpca:.1f}

## D4 noise robustness
- Correct (clean):   {correct_clean}/{n}
- Correct (noisy):   {correct_noisy}/{n}
- Interference rate:  {interference_rate}
- Flipped (right→wrong): {flipped}
- Gained  (wrong→right): {gained}

## Key observations
- **Query rewriting effect on D2:** Clean D2 changed from Week 3's 0.183 to {avg_d2_c:.3f}. Correct answers had D2={corr_d2:.3f} vs incorrect D2={incorr_d2:.3f}.
- **D4 interference:** {flipped} questions flipped from correct→wrong under noise. Interference rate = {interference_rate}.
- **Noise vs clean D2:** Avg D2 dropped from {avg_d2_c:.3f} (clean) to {avg_d2_n:.3f} (noisy) — noise degrades retrieval quality.

## Week 5 plan
- D6 difficulty interaction: seed dataset with single-hop low-distractor questions
- Plot accuracy degradation curve across difficulty axes
- Consider upgrading embedding model if D2 improvement is insufficient
"""

with open(PROJECT_ROOT / "notes/4_observations.md", "w") as f:
    f.write(notes)

print("=" * 65)
print("WEEK 4 COMPLETE")
print("=" * 65)
print()
print("Files created:")
print(f"  data/trajectories/4_agent_results_clean.json  ← {n} questions, clean, query rewriting")
print(f"  data/trajectories/4_agent_results_noisy.json  ← {n} questions, 25% noise injected")
print(f"  data/trajectories/4_d4_interference.json      ← D4 per-question + aggregate")
print(f"  notes/4_observations.md                        ← auto-generated observations")
print()
print("Commit:")
print("  git add .")
print("  git commit -m 'week4: query rewriting, D4 noise robustness, interference rate'")
print("  git push")

WEEK 4 COMPLETE

Files created:
  data/trajectories/4_agent_results_clean.json  ← 50 questions, clean, query rewriting
  data/trajectories/4_agent_results_noisy.json  ← 50 questions, 25% noise injected
  data/trajectories/4_d4_interference.json      ← D4 per-question + aggregate
  notes/4_observations.md                        ← auto-generated observations

Commit:
  git add .
  git commit -m 'week4: query rewriting, D4 noise robustness, interference rate'
  git push
